In [52]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, LeaveOneGroupOut
import homoglyphs as hg
import re
import string
import unicodedata
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import ParameterGrid

In [38]:
LING_SPAM_PATH = './datasets/Ling-Spam.csv'
NAZARIO_PATH = './datasets/Nazario.csv'
NIGERIAN_FRAUD_PATH = './datasets/Nigerian_Fraud.csv'
SPAMASSASSIN_PATH = './datasets/SpamAssasin.csv'

REQUIRED_COLUMNS = ['body', 'label']

DATASETS = {
    "Ling-Spam": LING_SPAM_PATH,
    "Nazario": NAZARIO_PATH,  # not using Nazario_5
    "Nigerian_Fraud": NIGERIAN_FRAUD_PATH,
    "SpamAssasin": SPAMASSASSIN_PATH
}

# Set random seed for reproducibility
np.random.seed(42)

# Load the dataset individually
#df = pd.read_csv(DATASETS["Ling-Spam"])

#df.head()

In [39]:
# Check for missing values in each dataset and the different datatypes of the columns
for index, name in DATASETS.items():
    print(f"Dataset: {name}")
    df = pd.read_csv(DATASETS[index])
    print(f"{df.isnull().sum()} + \n")
    print(f"{df.info()}" + "\n")

    # Body and label columns are the only common columns across all datasets, so we will only keep those columns for further analysis

Dataset: ./datasets/Ling-Spam.csv
subject    62
message     0
label       0
dtype: int64 + 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2893 entries, 0 to 2892
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   subject  2831 non-null   object
 1   message  2893 non-null   object
 2   label    2893 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 67.9+ KB
None

Dataset: ./datasets/Nazario.csv
sender       0
receiver    96
date         1
subject      4
body         0
urls         0
label        0
dtype: int64 + 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1565 entries, 0 to 1564
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   sender    1565 non-null   object
 1   receiver  1469 non-null   object
 2   date      1564 non-null   object
 3   subject   1561 non-null   object
 4   body      1565 non-null   object
 5   urls      1565 non-null   in

In [40]:
# Check the distribution of labels in each dataset
for index, name in DATASETS.items():
    print(f"Dataset: {name}")
    df = pd.read_csv(DATASETS[index])
    print(f"{df['label'].value_counts(dropna=False)}" + "\n")

Dataset: ./datasets/Ling-Spam.csv
label
0    2412
1     481
Name: count, dtype: int64

Dataset: ./datasets/Nazario.csv
label
1    1565
Name: count, dtype: int64

Dataset: ./datasets/Nigerian_Fraud.csv
label
1    3332
Name: count, dtype: int64

Dataset: ./datasets/SpamAssasin.csv
label
0    4091
1    1718
Name: count, dtype: int64



In [41]:
# Cleaning the SpamAssassin dataset
def clean_spam_assassin(spam_assassin_df):
    if 'body' in spam_assassin_df.columns:
        spam_assassin_df['body'] = spam_assassin_df['body'].fillna('')
    return spam_assassin_df


# Cleaning the Ling-Spam dataset
def clean_ling_spam(ling_spam_df):
    if 'message' in ling_spam_df.columns:
        ling_spam_df = ling_spam_df.rename(columns={'message': 'body'})
    return ling_spam_df



In [42]:
raw_datasets = {}
cleaned_datasets = {}

def preprocess_datasets():
    for name, path in DATASETS.items():
        df = pd.read_csv(path)
        raw_datasets[name] = df

    for name, df in raw_datasets.items():
        df = clean_spam_assassin(df) if name == "SpamAssasin" else df
        df = clean_ling_spam(df) if name == "Ling-Spam" else df

        for column in REQUIRED_COLUMNS:
            if column not in df.columns:
                raise ValueError(f"Column '{column}' is missing in dataset '{name}'")

            cleaned_datasets[name] = df[REQUIRED_COLUMNS].copy()

    for name, df in cleaned_datasets.items():
        print(f"Dataset: {name}")
        print(f"{df.info()}" + "\n")


preprocess_datasets()


Dataset: Ling-Spam
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2893 entries, 0 to 2892
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   body    2893 non-null   object
 1   label   2893 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 45.3+ KB
None

Dataset: Nazario
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1565 entries, 0 to 1564
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   body    1565 non-null   object
 1   label   1565 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 24.6+ KB
None

Dataset: Nigerian_Fraud
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3332 entries, 0 to 3331
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   body    3332 non-null   object
 1   label   3332 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 52.2+ KB
None

Dataset: SpamAssasi

In [43]:
# Training datasets for cross-dataset evaluation design using leave-one-dataset-out evaluation strategy. The training datasets are created by combining three datasets and the testing dataset is the remaining one.

# TF-IDF vectorization parameters
#analyzer = ['word', 'char_wb']
#ngram_range = [(1, 2), (3, 5)] # word-level and char-level
#max_features= [50000, 200000] # word-level and char-level
TFIDF_PARAMS = {
    "analyzer": "word",
    "ngram_range": (1, 2),
    "min_df": 2,
    "max_df": 0.5,
    "sublinear_tf": True,
    "dtype": np.float32,
    "max_features": 50000,
}

# XGBoost classifier parameters
learning_rate = [0.01, 0.05, 0.3]

XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "n_estimators": 500,
    "max_depth": 5,
    "max_bin": 128,
    "tree_method": "hist",
    "device": "cpu",
    "n_jobs": -1,
    "random_state": 42,
}

parameter_settings = {
    #'tfidf__analyzer': analyzer,
    #'tfidf__ngram_range': ngram_range,
    #'#tfidf__max_features': max_features,
    'classifier__learning_rate': learning_rate,
}


def make_model():
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            **TFIDF_PARAMS
        )),
        ('classifier', XGBClassifier(
            **XGB_PARAMS))
    ], memory=None)


baseline_model = make_model()

In [44]:
# PS: Some overlapping body content between the training and testing datasets may exist, which may lead to data leakage and overestimation of the model's performance.
def normalize_body(text):
    return re.sub(r'\s+', ' ', str(text)).lower().strip()

def remove_duplicates_in_dataset(df, dataset_name):
    result = df.copy()

    result["normalized_body"] = result["body"].map(normalize_body)

    # Remove empty bodies
    result = result[result["normalized_body"] != ""].copy()

    # Check duplicate bodies with different labels before removing duplicates.
    conflicting_bodies = result.groupby("normalized_body")["label"].nunique()

    print(f"Number of conflicting bodies in {dataset_name}: {conflicting_bodies[conflicting_bodies > 1].shape[0]}")

    # Keep copy of each normalized body
    result = result.drop_duplicates(subset="normalized_body", keep="first").reset_index(drop=True) # within this dataset

    return result


deduplicated_datasets = {}

for name, df in cleaned_datasets.items():
    deduplicated_datasets[name] = remove_duplicates_in_dataset(df, name)

    dedup_df = deduplicated_datasets[name]
    print(f"Dataset: {name}")
    print(f"Label counts: {dedup_df['label'].value_counts()}")
    print(f"Shape: {dedup_df.shape}")



Number of conflicting bodies in Ling-Spam: 0
Dataset: Ling-Spam
Label counts: label
0    2401
1     458
Name: count, dtype: int64
Shape: (2859, 3)
Number of conflicting bodies in Nazario: 0
Dataset: Nazario
Label counts: label
1    1526
Name: count, dtype: int64
Shape: (1526, 3)
Number of conflicting bodies in Nigerian_Fraud: 0
Dataset: Nigerian_Fraud
Label counts: label
1    3270
Name: count, dtype: int64
Shape: (3270, 3)
Number of conflicting bodies in SpamAssasin: 0
Dataset: SpamAssasin
Label counts: label
0    4090
1    1694
Name: count, dtype: int64
Shape: (5784, 3)


In [45]:
# Using a pipeline with TfidfVectorizer and XGBClassifier for the baseline model.

# 4-fold structure partitioning the benign side of Ling-Spam and SpamAssassin to be used for training data on test phishing datasets.

ling_dataset = deduplicated_datasets["Ling-Spam"].copy()
spam_assassin = deduplicated_datasets["SpamAssasin"].copy()
nigerian_fraud = deduplicated_datasets["Nigerian_Fraud"].copy()
nazario = deduplicated_datasets["Nazario"].copy()

ling_benign = ling_dataset[ling_dataset['label'] == 0].copy()
ling_spam = ling_dataset[ling_dataset['label'] == 1].copy()

spam_assassin_benign = spam_assassin[spam_assassin['label'] == 0].copy()
spam_assassin_spam = spam_assassin[spam_assassin['label'] == 1].copy()

# 80 20 train-test split for the benign side of spam datasets
ling_benign_train, ling_benign_test = train_test_split(ling_benign, test_size=0.2, random_state=42)
spam_assassin_benign_train, spam_assassin_benign_test = train_test_split(spam_assassin_benign, test_size=0.2,
                                                                         random_state=42)

ling_benign_train = ling_benign_train.assign(source="Ling-Spam")
ling_benign_test = ling_benign_test.assign(source="Ling-Spam")
ling_spam = ling_spam.assign(source="Ling-Spam")
spam_assassin_benign_train = spam_assassin_benign_train.assign(source="SpamAssassin")
spam_assassin_benign_test = spam_assassin_benign_test.assign(source="SpamAssassin")
spam_assassin_spam = spam_assassin_spam.assign(source="SpamAssassin")

nazario = nazario.assign(source="Nazario")
nigerian_fraud = nigerian_fraud.assign(source="Nigerian_Fraud")

# combined benign training and testing datasets
benign_train = pd.concat(
    [
        ling_benign_train,
        spam_assassin_benign_train
    ],
    ignore_index=True
)

benign_test = pd.concat(
    [
        ling_benign_test,
        spam_assassin_benign_test
    ],
    ignore_index=True
)

# Build custom four-fold LODO evaluation structure for cross-dataset evaluation design
folds = {
    "Ling-Spam": {
        "train": pd.concat(
            [
                spam_assassin_benign_train,
                nigerian_fraud,
                nazario
            ],
            ignore_index=True,
        ),
        "test": ling_dataset,
    },
    "SpamAssassin": {
        "train": pd.concat(
            [
                ling_benign_train,
                nigerian_fraud,
                nazario
            ],
            ignore_index=True,
        ),
        "test": spam_assassin
    },
    "Nigerian_Fraud": {
        "train": pd.concat(
            [
                benign_train,
                nazario
            ],
            ignore_index=True
        ),

        "test": pd.concat([
            benign_test,
            nigerian_fraud
        ],
            ignore_index=True)
    },
    "Nazario": {
        "train": pd.concat(
            [
                benign_train,
                nigerian_fraud
            ], ignore_index=True

        ),
        "test": pd.concat(
            [
                benign_test,
                nazario
            ],
            ignore_index=True
        ),
    }
}


folds["Ling-Spam"]["test"] = folds["Ling-Spam"]["test"].assign(source="Ling-Spam")
folds["SpamAssassin"]["test"] = folds["SpamAssassin"]["test"].assign(source="SpamAssassin")

In [54]:
# Evaluate the baseline model on each fold and print the results
baseline_results = []
for held_out_dataset_name, fold in folds.items():
    train_df = fold["train"]
    test_df = fold["test"]

    print(f"\nTraining on: {train_df['source'].unique()} | Testing on: {held_out_dataset_name}")
    print("\nTest distribution:")
    print(test_df["label"].value_counts().sort_index())

    # train the classifier
    for param in ParameterGrid(parameter_settings):
        model = make_model()
        model.set_params(**param)

        model.fit(train_df['body'], train_df['label'])

        pred = model.predict(test_df['body'])

        print("Parameters:")
        for p, value in param.items():
            print(f"{p}: {value}")
        print(f"Accuracy: {accuracy_score(test_df['label'], pred):.4f}")
        print(classification_report(
            test_df["label"],
            pred,
            zero_division=0
        ))
        print("Confusion matrix:")
        print(confusion_matrix(test_df["label"], pred))

        baseline_results.append({
            "test_dataset": held_out_dataset_name,
            "accuracy": accuracy_score(test_df["label"], pred)
        })

results_df = pd.DataFrame(baseline_results)
results_df

#testing_datasets = deduplicated_datasets.copy()


Training on: ['SpamAssassin' 'Nigerian_Fraud' 'Nazario'] | Testing on: Ling-Spam

Test distribution:
label
0    2401
1     458
Name: count, dtype: int64
Parameters:
classifier__learning_rate: 0.01
Accuracy: 0.4239
              precision    recall  f1-score   support

           0       0.84      0.38      0.53      2401
           1       0.16      0.63      0.26       458

    accuracy                           0.42      2859
   macro avg       0.50      0.51      0.39      2859
weighted avg       0.74      0.42      0.49      2859

Confusion matrix:
[[ 924 1477]
 [ 170  288]]
Parameters:
classifier__learning_rate: 0.05
Accuracy: 0.5327
              precision    recall  f1-score   support

           0       0.88      0.51      0.65      2401
           1       0.20      0.65      0.31       458

    accuracy                           0.53      2859
   macro avg       0.54      0.58      0.48      2859
weighted avg       0.77      0.53      0.59      2859

Confusion matrix:
[[1227 

,test_dataset,accuracy
0,Ling-Spam,0.423924
1,Ling-Spam,0.532704
2,Ling-Spam,0.533054
3,SpamAssassin,0.452801
4,SpamAssassin,0.615145
5,SpamAssassin,0.621542
6,Nigerian_Fraud,0.717006
7,Nigerian_Fraud,0.560954
8,Nigerian_Fraud,0.588094
9,Nazario,0.546903


In [47]:
label = r"[A-Za-z0-9](?:[A-Za-z0-9-]*[A-Za-z0-9])?"
segment = r"[A-Za-z0-9]+(?:\s*[._-]\s*[A-Za-z0-9]+)*"

url_pattern = re.compile(
    rf"""
    (?:
        https?\s*:\s*/\s*/
        |
        www\s*\.\s*
    )
    {label}
    (?:\s*\.\s*{label})+
    (?:\s*/\s*{segment})*
    (?:\s*[?&]\s*[A-Za-z0-9._~:/?#\[\]@!$'()*+,;=%-]+)?
    """,
    re.IGNORECASE | re.VERBOSE
)


def normalize_url(url):
    url = re.sub(r"\s*([:/?.=&])\s*", r"\1", url)
    return url.rstrip(".,;:!?)]}")


# Function to extract URLs in the email body
def extract_url(text):
    normalized_url = [normalize_url(url_match.group(0)) for url_match in url_pattern.finditer(str(text))]
    print(normalized_url)
    return normalized_url

In [48]:
# Perturbation functions for homoglyph substitution, and encoding obfuscation. These functions will be used to generate adversarial examples for the email text data in the respective test datasets.

# Create a Homoglyphs object with the desired categories
homoglyphs = hg.Homoglyphs(categories=("LATIN", "CYRILLIC"))

latin_letters = string.ascii_letters

# map of similar looking latin letters to cyrillic letters
latin_to_cyrillic = {}

for latin in latin_letters:
    combinations = homoglyphs.get_combinations(latin)

    for c in combinations:
        name = unicodedata.name(c, "")
        if name and "CYRILLIC" in name:
            latin_to_cyrillic[latin] = c
            break

print(latin_to_cyrillic)


def replace_url(m):
    url = m.group(0)
    # Replace each character in the URL with its cyrillic equivalent, if available
    replaced_url = "".join(latin_to_cyrillic.get(char, char) or char for char in url)
    return replaced_url


def homoglyph_substitution(text):
    url = str(text)
    substituted_text = url_pattern.sub(replace_url, url)

    return substituted_text

{'a': 'а', 'b': 'Ь', 'c': 'с', 'd': 'ԁ', 'e': 'е', 'h': 'һ', 'i': 'і', 'j': 'ј', 'l': 'І', 'o': 'о', 'p': 'р', 'q': 'ԛ', 'r': 'г', 's': 'ѕ', 'v': 'ѵ', 'w': 'ѡ', 'x': 'х', 'y': 'у', 'A': 'А', 'B': 'В', 'C': 'С', 'E': 'Е', 'G': 'Ԍ', 'H': 'Н', 'I': 'І', 'J': 'Ј', 'K': 'К', 'M': 'М', 'O': 'О', 'P': 'Р', 'S': 'Ѕ', 'T': 'Т', 'V': 'Ѵ', 'W': 'Ԝ', 'X': 'Х', 'Y': 'У'}


In [49]:
#homoglyph_substitution("Hello, check out this link: http : / / www . 123net . co . uk / agents / prebuild . htm and also visit www.testsite.org for more info.")

In [50]:
# Add perturbation to test datasets
#perturbed_testing_datasets = {}
#for name, df in testing_datasets.items():
#    df['body_homoglyph'] = (df['body'].fillna("").apply(homoglyph_substitution))
#    perturbed_testing_datasets[name] = df
#    print(f"{name}: {df.shape}")


In [51]:
# first test performance on the original test datasets, then on the perturbed test datasets. The training datasets remain unchanged. this is to have a baseline performance on the original datasets to compare with.
